<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day4/notebooks/4_llm_access_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 4 (extra): Reaching an LLM Three Ways

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

A short demonstration to go with the *Inside GPT* slides. We take **one small open model**,
Qwen2.5 1.5B Instruct, the same model as in the annotation notebook, and **one small task**:
coding three bill titles by policy area, the task you know from Day 3. Then we reach the model
in three different ways:

| | Route | Where the model runs | What you need |
|---|---|---|---|
| 1 | **Hugging Face** `transformers` | in your Python session (here: Colab's GPU) | a GPU for anything sizable |
| 2 | **Ollama** | in a separate app on the same machine | a laptop, or Colab |
| 3 | **OpenRouter** | on a provider's servers | an account and an API key |

Along the way we look at things the slides talked about: tokens, the chat template, the
probabilities behind the next token, and temperature.

**Time:** about 30 minutes. Section 3 needs an OpenRouter API key; without one, its cells
simply skip themselves.

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

## 0. Setup

In [ ]:
!pip install -q transformers accelerate openai ollama

In [ ]:
import os
import time
import subprocess

import requests
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)
pd.set_option("display.max_colwidth", 120)

print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

### The task

The same three bill titles go to every route, with the same instructions. The instructions go
into a **system** message, the title into a **user** message.

In [ ]:
TITLES = [
    "A bill to increase funding for public schools and hire more teachers.",
    "To provide additional health care benefits to veterans exposed to burn pits.",
    "To repeal the estate tax.",
]
LABELS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]

SYSTEM = ("You code US bill titles by policy area. Answer with exactly one of: "
          + ", ".join(LABELS) + ". Reply with the label only.")


def make_messages(title):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Bill title: {title}"},
    ]


results = []   # one row per route and title, compared at the end
make_messages(TITLES[0])

---
## 1. Hugging Face: the model inside your Python session

Exactly as on Day 3: download the weights from the Hub and load them. This time we load a
*causal* language model (`AutoModelForCausalLM`), the GPT kind, instead of BERT. At about
3 GB in 16-bit, the download takes a minute.

In [ ]:
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
model = AutoModelForCausalLM.from_pretrained(HF_MODEL, dtype="auto", device_map="auto")

print(f"{model.num_parameters() / 1e9:.2f} billion parameters, vocabulary of {len(tokenizer):,} tokens")

### 1.1 Tokens

GPT-style models use byte-pair encoding, a cousin of BERT's WordPiece. The `Ġ` you'll see marks
a space before the token, which is how this tokenizer keeps track of where words begin.

In [ ]:
for text in ["Wageningen University is in the Netherlands.",
             "Wageningen University ligt in Nederland."]:
    pieces = tokenizer.tokenize(text)
    print(f"{len(text.split())} words → {len(pieces)} tokens: {pieces}")

### 1.2 What the model actually sees

The model never sees our list of messages. The **chat template** turns it into one long text
with special tokens, and the model simply continues that text.

In [ ]:
print(tokenizer.apply_chat_template(make_messages(TITLES[0]), tokenize=False, add_generation_prompt=True))

`add_generation_prompt=True` adds the last line, `<|im_start|>assistant`: the cue that it's the
model's turn. Now watch what happens when we *don't* give a system message:

In [ ]:
print(tokenizer.apply_chat_template([{"role": "user", "content": "Hello!"}],
                                   tokenize=False, add_generation_prompt=True))

A system message appears that we never wrote. It's built into this model's template. Other
model families do this differently, which is one more reason to print the template once for
any model you use in research.

### 1.3 The probabilities behind the next token

One pass through the model gives a probability for **every** token in the vocabulary. These are
the ten most likely first tokens of the answer for our first title:

In [ ]:
prompt = tokenizer.apply_chat_template(make_messages(TITLES[0]), tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    next_token_scores = model(input_ids=inputs["input_ids"],
                              attention_mask=inputs["attention_mask"]).logits[0, -1]   # scores for the NEXT token only
probs = torch.softmax(next_token_scores.float(), dim=-1)
top = torch.topk(probs, 10)

pd.DataFrame({
    "token": [repr(tokenizer.decode(int(i))) for i in top.indices],
    "probability": top.values.cpu().numpy().round(3),
})

Generating an answer means repeating this: pick a token, add it to the text, and run the model
again, until it produces the end token `<|im_end|>`.

### 1.4 Coding the titles

`do_sample=False` always takes the most likely token (the same as temperature 0). You may see a
warning that the model's default `temperature` is ignored: that's what we want here.

In [ ]:
def hf_chat(messages, temperature=0.0, max_new_tokens=10):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    settings = {"do_sample": True, "temperature": temperature} if temperature > 0 else {"do_sample": False}
    with torch.no_grad():
        output = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **settings)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]          # keep only the answer
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


for title in TITLES:
    start = time.time()
    answer = hf_chat(make_messages(title))
    results.append({"route": "1 Hugging Face", "model": HF_MODEL, "title": title,
                    "answer": answer, "seconds": round(time.time() - start, 2)})
    print(f"{answer:<36} {title}")

### 1.5 Temperature

For coding data we want the same answer every time. For writing, variety can be useful.
Same request, three times at temperature 0 and three times at temperature 1:

In [ ]:
headline = [{"role": "user", "content": "Write a one-sentence news headline about Dutch dairy farmers "
                                        "protesting new nitrogen rules."}]

print("Temperature 0:")
for _ in range(3):
    print("  ", hf_chat(headline, temperature=0, max_new_tokens=30))

print("\nTemperature 1:")
for _ in range(3):
    print("  ", hf_chat(headline, temperature=1.0, max_new_tokens=30))

At temperature 0 the three answers are identical; at temperature 1 they differ. Keep that in
mind when you use an LLM as a measurement instrument.

Before we move on, we free the GPU memory for Ollama:

In [ ]:
del model
torch.cuda.empty_cache()

---
## 2. Ollama: a local app that serves models

[Ollama](https://ollama.com) is a free app that downloads open models and runs them on your own
computer. Python then talks to it over a local connection, so your texts never leave the
machine. **On your own laptop** you would:

1. download and install Ollama from [ollama.com](https://ollama.com),
2. open a terminal and run `ollama pull qwen2.5:1.5b`,
3. use the Python code from Section 2.2 onward, unchanged.

Colab is a Linux machine, so here we install the Linux version instead. That takes about a
minute.

In [ ]:
# Only needed in Colab or on a Linux server. On a laptop, install the app from ollama.com instead.
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

### 2.1 Start Ollama and download the model

On a laptop, the Ollama app starts its server for you. In Colab we start it ourselves, in the
background. Then we download the model: Ollama's version is **quantized to 4 bits**, about 1 GB
instead of 3.

In [ ]:
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):                     # wait until the server answers
    try:
        print(requests.get("http://localhost:11434", timeout=1).text)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

In [ ]:
!ollama pull qwen2.5:1.5b

### 2.2 Coding the titles

In [ ]:
import ollama

OLLAMA_MODEL = "qwen2.5:1.5b"

for title in TITLES:
    start = time.time()
    response = ollama.chat(model=OLLAMA_MODEL, messages=make_messages(title),
                           options={"temperature": 0})
    answer = response.message.content.strip()
    results.append({"route": "2 Ollama", "model": OLLAMA_MODEL, "title": title,
                    "answer": answer, "seconds": round(time.time() - start, 2)})
    print(f"{answer:<36} {title}")

### 2.3 Under the hood: it's just a web address

The `ollama` package is a thin wrapper. Ollama listens at `http://localhost:11434`, and any
program that can send a web request can use it. This is the same request without the package:

In [ ]:
reply = requests.post("http://localhost:11434/api/chat", json={
    "model": OLLAMA_MODEL,
    "messages": make_messages(TITLES[2]),
    "stream": False,                      # one complete answer instead of word by word
    "options": {"temperature": 0},
}).json()

print("Answer:          ", reply["message"]["content"])
print("Tokens generated:", reply["eval_count"])
print("Seconds:         ", round(reply["total_duration"] / 1e9, 2))

Did Ollama give the same answers as Hugging Face in Section 1? It's the same model, but here
it's quantized to 4 bits, so small differences are possible. That's worth knowing when you
report which model you used: the name alone doesn't say which version or precision ran.

---
## 3. OpenRouter: hosted models through one API

[OpenRouter](https://openrouter.ai) gives you one account, one API key and one interface for
hundreds of models from many providers, open and closed. You pay per token; for this notebook
that's a fraction of a cent. Some models have free variants (their names end in `:free`), but
those are rate-limited and come and go.

**Setting up a key:**

1. Make an account at [openrouter.ai](https://openrouter.ai) and create a key under *Keys*.
2. In Colab, click the 🔑 icon (*Secrets*) in the left sidebar, add a secret named
   `OPENROUTER_API_KEY` with your key as its value, and switch on notebook access.

Never paste a key into the notebook itself: notebooks get shared, keys shouldn't.

> ⚠️ **Your texts go to the provider** that runs the model, under their terms. Don't send
> anything sensitive.

In [ ]:
def get_secret(name):
    try:
        from google.colab import userdata          # Colab's Secrets
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)                # anywhere else: an environment variable

OPENROUTER_KEY = get_secret("OPENROUTER_API_KEY")
print("Key found." if OPENROUTER_KEY else "No key found: the OpenRouter cells below will skip themselves.")

### 3.1 What's on offer

The model list is public, so this works without a key. Prices are per **million** tokens.

In [ ]:
catalog = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]

models = pd.DataFrame([{
    "id": m["id"],
    "context window": m.get("context_length"),
    "$ per million input tokens": pd.to_numeric(m["pricing"]["prompt"], errors="coerce") * 1e6,
    "$ per million output tokens": pd.to_numeric(m["pricing"]["completion"], errors="coerce") * 1e6,
} for m in catalog])

print(f"{len(models)} models available today, {models['id'].str.endswith(':free').sum()} of them free variants")
models[models["id"].str.startswith("qwen/qwen-2.5")].sort_values("id")

### 3.2 Coding the titles with a bigger model

Same family, much bigger: Qwen2.5 **72B** instead of 1.5B. That's far too big for Colab's GPU,
so here someone else's servers do the work. OpenRouter speaks the same "language" as OpenAI's
API, so we use the `openai` package and only change the address.

Model names on OpenRouter change over time. If this one is gone, pick another from the table
above.

In [ ]:
from openai import OpenAI

OR_MODEL = "qwen/qwen-2.5-72b-instruct"

if not OPENROUTER_KEY:
    print("Skipped: no OpenRouter key.")
else:
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY)
    for title in TITLES:
        start = time.time()
        response = client.chat.completions.create(model=OR_MODEL, messages=make_messages(title),
                                                  temperature=0, max_tokens=10)
        answer = response.choices[0].message.content.strip()
        results.append({"route": "3 OpenRouter", "model": OR_MODEL, "title": title,
                        "answer": answer, "seconds": round(time.time() - start, 2)})
        print(f"{answer:<36} {title}")

    usage = response.usage
    print(f"\nLast request: {usage.prompt_tokens} tokens in, {usage.completion_tokens} tokens out")

Notice how many tokens went *in* compared with how many came out: the instructions are sent
again with every single title. With thousands of texts, the length of your prompt is what you
pay for.

What does that cost? A rough calculation for coding 10,000 titles with this prompt:

In [ ]:
prices = models.set_index("id")
if OR_MODEL not in prices.index:
    print(f"{OR_MODEL} is not in today's list: pick another model above.")
else:
    tokens_in, tokens_out = 60, 5        # per title, roughly: compare with the usage printed above
    cost = 10_000 * (tokens_in * prices.loc[OR_MODEL, "$ per million input tokens"]
                     + tokens_out * prices.loc[OR_MODEL, "$ per million output tokens"]) / 1e6
    print(f"About ${cost:.2f} for 10,000 titles with {OR_MODEL}")

---
## 4. Three routes, side by side

In [ ]:
table = pd.DataFrame(results)
table.pivot(index="title", columns="route", values="answer")

In [ ]:
table.groupby("route")[["seconds"]].mean().round(2).rename(columns={"seconds": "seconds per title"})

A few things to look for:

- **Do the routes agree?** The first two run the same model, in 16-bit and in 4-bit. The third
  runs a far larger relative. Where they disagree, which answer would a human coder give?
- **Speed** depends on the hardware and, for hosted models, on how busy the provider is.
  The first call to a model is often slower because it has to be loaded.
- **Three titles prove nothing.** To use any of these for research, code a few hundred texts by
  hand and measure agreement, as in the annotation notebook.

### What to report when you use an LLM

The exact model name *and* route (Hub name and revision, Ollama tag, or OpenRouter model and
provider), the precision (16-bit or quantized), the full prompt including any system message,
the settings (temperature, maximum tokens), the date, and how you validated the output.

In [ ]:
server.terminate()      # stop the Ollama server